# Understanding Runtime and Context

## Full example

In [12]:
import os
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model


_ = load_dotenv(find_dotenv())

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

# 1. Define the context schema (static user data)
@dataclass
class UserContext:
    user_id: str
    user_name: str
    preferred_language: str

# 2. Define tools that access the runtime context
@tool
def get_user_profile(runtime: ToolRuntime[UserContext]) -> str:
    """Look up the current user's profile information."""
    ctx = runtime.context
    return f"User ID: {ctx.user_id}, Name: {ctx.user_name}, Language: {ctx.preferred_language}"

@tool
def get_user_orders(runtime: ToolRuntime[UserContext]) -> str:
    """Look up the current user's recent orders."""
    # Simulated order data based on user_id
    orders = {
        "user_42": [{"id": "ORD-001", "item": "Wireless Mouse", "status": "Delivered"}],
        "user_99": [{"id": "ORD-100", "item": "Keyboard", "status": "Processing"}],
    }
    user_orders = orders.get(runtime.context.user_id, [])
    if not user_orders:
        return "No orders found."
    return f"Recent orders:\n  - {user_orders[0]['id']}: {user_orders[0]['item']} ({user_orders[0]['status']})"

# 3. Create the agent with a context_schema
model = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0, api_key=api_key, base_url=base_url)
ollama = init_chat_model("llama3.1:8b ", model_provider="ollama", temperature=0)

agent = create_agent(
    #model=model,
    model=ollama,
    tools=[get_user_profile, get_user_orders],
    system_prompt="You are a customer support assistant. Use the available tools to help the user.",
    context_schema=UserContext, # Tell the agent to expect UserContext
)

# 4. Invoke the agent with context for user_42
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Show me my profile and recent orders."}]},
    context=UserContext(user_id="user_42", user_name="Alice", preferred_language="English"), # Inject context
)
print(result["messages"][-1].content)

# 5. Invoke the agent with context for a different user
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Show me my profile and recent orders"}]},
    context=UserContext(user_id="user_99", user_name="Bob", preferred_language="Portuguese"), # Inject context
)
print(result["messages"][-1].content)

Based on the tool call responses, I can provide you with the following information:

**Your Profile:**

* User ID: user_42
* Name: Alice
* Language: English

**Recent Orders:**

* ORD-001: Wireless Mouse (Delivered)
Based on the tool call responses, I can provide you with the following information:

**Your Profile:**

* User ID: user_99
* Name: Bob
* Language: Portuguese

**Recent Orders:**

* ORD-100: Keyboard (Processing)

Please let me know if you would like to view more details about your orders or profile.


## Creating context (data schema)

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "آبی"
    least_favourite_colour: str = "زرد"

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    #model="gpt-5-nano",
    model=ollama,
    context_schema=ColourContext  
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="رنگ مورد علاقه من چیه؟")]},
    context=ColourContext()
)

In [5]:
print(response["messages"][-1].content)

من نمی دونم رنگ مورد علاقه شما چیه.


## Accessing Context

In [6]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [8]:
agent = create_agent(
    model=ollama,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="رنگ مورد علاقه ام چیه؟")]},
    context=ColourContext()
)

print(response["messages"][-1].content)

c:\Users\meisa\Desktop\Code\ML\NLP\HuggingFace_course\.env\lib\site-packages\pydantic\functional_validators.py:839: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='زرد'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\meisa\Desktop\Code\ML\NLP\HuggingFace_course\.env\lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='زرد'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


خوب است که رنگ مورد علاقه شما آبی است. آیا می‌خواهید درباره رنگ‌های دیگر صحبت کنیم؟


In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="رنگ مورد علاقه ام چیه؟")]},
    context=ColourContext(favourite_colour="green")
)

print(response["messages"][-1].content)

c:\Users\meisa\Desktop\Code\ML\NLP\HuggingFace_course\.env\lib\site-packages\pydantic\functional_validators.py:839: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='زرد'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\meisa\Desktop\Code\ML\NLP\HuggingFace_course\.env\lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='زرد'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


رنگ مورد علاقه شما سبز است.


## Dynamic System Prompt

In [13]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from typing import TypedDict

class UserContext(TypedDict):
    user_level: str  # "beginner" یا "expert"


In [14]:
@dynamic_prompt
def adaptive_prompt(request: ModelRequest) -> str:
    """System prompt بر اساس سطح کاربر"""
    level = request.runtime.context.get("user_level", "beginner")
    
    if level == "expert":
        return "You are a technical assistant. Use precise terminology and assume advanced knowledge."
    else:
        return "You are a friendly assistant. Explain concepts simply and avoid jargon."


In [15]:
agent_adaptive = create_agent(
    model=ollama,
    middleware=[adaptive_prompt],
    context_schema=UserContext,
)

In [16]:
# برای مبتدی
response_beginner = agent_adaptive.invoke(
    {"messages": [{"role": "user", "content": "What is machine learning?"}]},
    context={"user_level": "beginner"}
)
print("For beginner:")
print(response_beginner["messages"][-1].content)


For beginner:
Machine learning is a way that computers can learn and get better at doing things on their own.

Imagine you're teaching a child to recognize different animals. At first, they might not know what a cat or a dog looks like, but as you show them many pictures and tell them what they are, they start to get better at recognizing them.

Machine learning is similar. Computers are given lots of examples of things, like pictures or words, and they learn to recognize patterns and make predictions based on what they've seen. The more examples they get, the better they become at doing things like:

* Recognizing faces or objects in pictures
* Understanding what people are saying in different languages
* Predicting what you might like to buy or watch based on what you've liked before

The magic of machine learning is that it can get better and better over time, without being explicitly programmed to do so. It's like the computer is learning from experience, just like a child does!

D

In [17]:
# برای متخصص
response_expert = agent_adaptive.invoke(
    {"messages": [{"role": "user", "content": "What is machine learning?"}]},
    context={"user_level": "expert"}
)
print("For expert:")
print(response_expert["messages"][-1])


For expert:
content="Machine learning (ML) is a subfield of artificial intelligence (AI) that enables systems to automatically improve their performance on a task without being explicitly programmed. It involves developing algorithms and statistical models that can learn from data, identify patterns, and make predictions or decisions based on that data.\n\nThere are several key aspects of machine learning:\n\n1. **Data-driven**: Machine learning relies on large datasets to train and validate models. The quality and quantity of the data are crucial for the model's performance.\n2. **Pattern recognition**: ML algorithms identify patterns and relationships within the data, which enables them to make predictions or decisions.\n3. **Model optimization**: The goal of machine learning is to optimize the model's performance on a specific task, such as classification, regression, or clustering.\n4. **Generalization**: A well-trained ML model should be able to generalize its knowledge to new, un